# Regularisation — Logistic Regression

Same idea as Ridge/Lasso, applied to classification. Penalise large coefficients to prevent overfitting.

**sklearn's parameter is `C`, not `alpha`:**

```
C = 1 / λ

Small C  → strong regularisation  (large λ)
Large C  → weak regularisation    (small λ)
```

**Penalty types:**
- `penalty='l2'` — Ridge-like (default)
- `penalty='l1'` — Lasso-like (feature selection)
- `penalty='elasticnet'` — mix of both
- `penalty=None` — no regularisation

---

## Pipeline

1. Generate synthetic classification data (self-contained)
2. Train Logistic Regression with default C
3. Sweep C — observe coefficient size, train/test accuracy
4. Compare L1 vs L2 penalty
5. GridSearchCV to find the best C

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

np.random.seed(42)

## Step 1 — Generate Data

- 200 samples, 20 features
- Only 5 informative, 15 noise
- Binary target (0 / 1)

In [ ]:
X, y = make_classification(
    n_samples=200,
    n_features=20,
    n_informative=5,
    n_redundant=2,
    n_classes=2,
    random_state=42
)

print('X shape:', X.shape)
print('y distribution:', np.bincount(y))

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)
print('Train:', X_train.shape, '  Test:', X_test.shape)

## Step 2 — Default Logistic Regression (C=1.0, L2)

By default sklearn applies L2 regularisation with C=1.0.

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

print(f'Train accuracy: {model.score(X_train, y_train):.3f}')
print(f'Test  accuracy: {model.score(X_test, y_test):.3f}')
print(f'Sum of |coefficients|: {np.sum(np.abs(model.coef_)):.2f}')
print(f'Largest |coefficient|: {np.max(np.abs(model.coef_)):.2f}')

## Step 3 — Sweep C (L2 Regularisation)

Watch how coefficients shrink and train/test accuracy changes as we vary C.

In [ ]:
Cs = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
rows = []
for C in Cs:
    m = LogisticRegression(C=C, penalty='l2', max_iter=1000).fit(X_train, y_train)
    rows.append({
        'C': C,
        'sum_|coef|': round(np.sum(np.abs(m.coef_)), 2),
        'train_acc': round(m.score(X_train, y_train), 3),
        'test_acc':  round(m.score(X_test,  y_test),  3)
    })
print(pd.DataFrame(rows))

**Pattern:**
- Small C → strong regularisation → coefficients tiny → may underfit
- Large C → weak regularisation → coefficients large → may overfit
- Best C → balance between train and test accuracy

---

## Step 4 — L1 (Lasso) vs L2 (Ridge) Penalty

L1 zeroes out unimportant features — feature selection. L2 just shrinks them.

In [ ]:
# L1 needs solver='liblinear' or 'saga'
l1 = LogisticRegression(C=0.1, penalty='l1', solver='liblinear').fit(X_train, y_train)
l2 = LogisticRegression(C=0.1, penalty='l2').fit(X_train, y_train)

print('L1 — non-zero coefficients:', np.sum(l1.coef_ != 0), '/ 20')
print('L2 — non-zero coefficients:', np.sum(l2.coef_ != 0), '/ 20')
print()
print(f'L1 test accuracy: {l1.score(X_test, y_test):.3f}')
print(f'L2 test accuracy: {l2.score(X_test, y_test):.3f}')

In [ ]:
comparison = pd.DataFrame({
    'feature': [f'X{i}' for i in range(20)],
    'L1_coef': np.round(l1.coef_[0], 3),
    'L2_coef': np.round(l2.coef_[0], 3)
})
print(comparison)

Notice L1 has many exact zeros — features dropped from the model. L2 keeps all features but with small coefficients.

---

## Step 5 — GridSearchCV: Find the Best C

Use cross-validation to pick C robustly.

In [ ]:
param_grid = {
    'C':       [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l1', 'l2']
}

grid = GridSearchCV(
    LogisticRegression(solver='liblinear', max_iter=1000),
    param_grid,
    cv=5,
    scoring='accuracy'
)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print(f'Best CV accuracy: {grid.best_score_:.3f}')
print(f'Held-out test accuracy: {grid.score(X_test, y_test):.3f}')

GridSearchCV chose the best combination of C and penalty using 5-fold cross-validation on the training set, then we evaluate once on the held-out test set.

---

## Summary

| | Logistic Regression |
|-|---------------------|
| Regularisation parameter | `C` (inverse of λ) |
| Small C | Strong regularisation, smaller coefficients |
| Large C | Weak regularisation, model behaves like plain LR |
| `penalty='l1'` | Lasso-like — feature selection |
| `penalty='l2'` | Ridge-like — coefficient shrinkage (default) |
| Choose C / penalty | Via `GridSearchCV` with CV |

> Always standardise features. Always tune C with cross-validation — never pick by hand on a single split.